In [1]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer


class EEGTextMetaDataset(Dataset):
    def __init__(self, eeg_dir, metadata_dir, tokenizer, max_length=64, use_emotional_tone=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_emotional_tone = use_emotional_tone

        # -------------------------
        # 1. Load EEG files (all subjects)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        self.eeg_file_paths = eeg_files
        self.eeg_data_list = []
        self.index_map = []

        for subj_idx, path in enumerate(self.eeg_file_paths):
            eeg = np.load(path, mmap_mode='r')
            assert eeg.ndim == 3 and eeg.shape[1:] == (62, 400), \
                f"EEG file {path} has shape {eeg.shape}, expected (*, 62, 400)"
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        print(f"Found {len(self.eeg_file_paths)} EEG files → Total samples: {total_samples}")

        # -------------------------
        # 2. Load Metadata JSONs
        # -------------------------
        metadata_files = sorted(
            [os.path.join(dp, f)
             for dp, dn, filenames in os.walk(metadata_dir)
             for f in filenames if f.endswith(".json")]
        )
        if not metadata_files:
            raise FileNotFoundError(f"No metadata JSON files found in {metadata_dir}")

        self.metadata_list = []
        for fpath in metadata_files:
            with open(fpath, 'r', encoding='utf-8') as f:
                meta = json.load(f)

                # --- Assertion for essential content ---
                assert "semantic_features" in meta and "scene_category" in meta["semantic_features"], \
                    f"Missing scene_category in {fpath}"
                assert "visual_attributes" in meta and "major_colors" in meta["visual_attributes"], \
                    f"Missing major_colors in {fpath}"
                
                # Robustly handle missing 'objects' key
                if "objects" not in meta.get("semantic_features", {}):
                    meta["semantic_features"]["objects"] = []

                self.metadata_list.append(meta)

        base_count = len(self.metadata_list)
        print(f"Loaded {base_count} metadata JSON files")

        # -------------------------
        # 3. Build base captions
        # -------------------------
        base_captions = []
        for meta in self.metadata_list:
            caption_text = meta["caption"]["text"]
            if self.use_emotional_tone and "emotional_tone" in meta["caption"]:
                caption_text += f". Tone: {meta['caption']['emotional_tone']}"
            base_captions.append(caption_text)

        # Repeat for each subject
        num_subjects = len(self.eeg_file_paths)
        self.captions = base_captions * num_subjects
        self.metadata_repeated = self.metadata_list * num_subjects

        assert len(self.captions) == len(self.metadata_repeated) == len(self.index_map), \
            "Mismatch after repeating captions and metadata for subjects"

        # -------------------------
        # 4. Encode metadata categories (Scene, Color, and Objects)
        # -------------------------
        scene_categories = sorted(list({m["semantic_features"]["scene_category"] for m in self.metadata_list}))
        colors = sorted(list({m["visual_attributes"]["major_colors"][0]["color"].split()[0]
                               for m in self.metadata_list}))
        all_objects = set()
        for m in self.metadata_list:
            all_objects.update(m["semantic_features"]["objects"])
        objects_vocab = sorted(list(all_objects))

        self.scene_to_id = {scene: i for i, scene in enumerate(scene_categories)}
        self.color_to_id = {c: i for i, c in enumerate(colors)}
        self.object_to_id = {obj: i for i, obj in enumerate(objects_vocab)}
        
        # Create inverse mapping for easier lookup
        self.id_to_scene = {i: scene for scene, i in self.scene_to_id.items()}
        self.id_to_color = {i: c for c, i in self.color_to_id.items()}
        self.id_to_object = {i: obj for obj, i in self.object_to_id.items()}

        print(f"Scene categories: {len(self.scene_to_id)} | Colors: {len(self.color_to_id)} | Objects: {len(self.object_to_id)}")
    
    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        subj_idx, local_idx = self.index_map[idx]
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        caption = self.captions[idx]
        
        tokenized = self.tokenizer(
            caption, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        
        # This is the line from your CORRECT __getitem__ method
        input_ids = tokenized['input_ids'].squeeze(0)
        
        meta = self.metadata_repeated[idx]
        
        scene_id = self.scene_to_id[meta["semantic_features"]["scene_category"]]
        color_id = self.color_to_id[meta["visual_attributes"]["major_colors"][0]["color"].split()[0]]
        scalar_meta_tensor = torch.tensor([scene_id, color_id], dtype=torch.float32)

        objects_present = meta["semantic_features"]["objects"]
        object_multi_hot = torch.zeros(len(self.object_to_id), dtype=torch.float32)
        for obj in objects_present:
            if obj in self.object_to_id:
                obj_id = self.object_to_id[obj]
                object_multi_hot[obj_id] = 1.0
        
        metadata_tensor = torch.cat((scalar_meta_tensor, object_multi_hot))
        
        # This is the CORRECT return statement
        return eeg_tensor, input_ids, metadata_tensor

In [2]:
import h5py
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from tqdm.auto import tqdm

# This assumes the EEGTextMetaDataset class definition from our previous conversation
# is in the same file or has been imported.

# --- Paths ---
HDF5_FILE = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
EEG_DIR = "/home/poorna/data/preprocessed_eeg"
METADATA_DIR = "/home/poorna/data/metadata_dir"

# --- Dataset ---
print("Initializing dataset...")
tokenizer = BertTokenizer.from_pretrained("/home/poorna/models/bert-base-uncased")

# Make sure you are using the corrected Dataset class from the previous step
dataset = EEGTextMetaDataset(EEG_DIR, METADATA_DIR, tokenizer)
loader = DataLoader(dataset, batch_size=1, shuffle=False) # batch=1 for sequential save

# --- Get dynamic shapes from a sample ---
# This makes the code robust to changes in the dataset's output shapes.
n_samples = len(dataset)
# MODIFIED: The dataset now returns the input_ids tensor directly as the second item
sample_eeg, sample_input_ids, sample_meta = dataset[0]

# --- Create HDF5 file ---
print(f"Creating HDF5 file at {HDF5_FILE}...")
with h5py.File(HDF5_FILE, "w") as f:
    # Shapes are determined dynamically from the actual dataset output
    eeg_shape = (n_samples, *sample_eeg.shape)
    # MODIFIED: Get the shape directly from the sample_input_ids tensor
    token_shape = (n_samples, *sample_input_ids.shape)
    meta_shape = (n_samples, *sample_meta.shape)

    print(f"Allocating space for {n_samples} samples...")
    print(f"  - EEG shape: {eeg_shape}")
    print(f"  - Token shape: {token_shape}")
    print(f"  - Metadata shape: {meta_shape}")

    eeg_ds = f.create_dataset("eeg", shape=eeg_shape, dtype="float32")
    tokens_ds = f.create_dataset("input_ids", shape=token_shape, dtype="int64")
    meta_ds = f.create_dataset("metadata", shape=meta_shape, dtype="float32")

    # Iterate and save directly to file (low memory usage)
    print("Writing data to HDF5 file...")
    # MODIFIED: Renamed the loop variable for clarity (tokenized -> input_ids_tensor)
    for idx, (eeg_tensor, input_ids_tensor, metadata_tensor) in enumerate(tqdm(loader, desc="Saving to HDF5")):
        # The squeeze(0) is needed because the DataLoader adds a batch dimension of 1
        eeg_ds[idx] = eeg_tensor.squeeze(0).numpy()
        # MODIFIED: Use the input_ids_tensor directly
        tokens_ds[idx] = input_ids_tensor.squeeze(0).numpy()
        meta_ds[idx] = metadata_tensor.squeeze(0).numpy()

print(f"\nSuccessfully saved dataset to {HDF5_FILE}")

Initializing dataset...
Found 20 EEG files → Total samples: 28000
Loaded 1400 metadata JSON files
Scene categories: 77 | Colors: 53 | Objects: 61
Creating HDF5 file at /home/poorna/data/eeg_dataset_with_objects_reduced.h5...
Allocating space for 28000 samples...
  - EEG shape: (28000, 62, 400)
  - Token shape: (28000, 64)
  - Metadata shape: (28000, 63)
Writing data to HDF5 file...


Saving to HDF5:   0%|          | 0/28000 [00:00<?, ?it/s]


Successfully saved dataset to /home/poorna/data/eeg_dataset_with_objects_reduced.h5


In [3]:
import h5py
import torch

with h5py.File("/home/poorna/data/eeg_dataset_with_objects_reduced.h5", "r") as f:
    print(list(f.keys()))  # ['eeg', 'input_ids', 'metadata']
    eeg_sample = torch.tensor(f["eeg"][0])         # first sample EEG tensor
    tokens_sample = torch.tensor(f["input_ids"][0])
    meta_sample = torch.tensor(f["metadata"][0])

print(eeg_sample.shape, tokens_sample.shape, meta_sample.shape)

['eeg', 'input_ids', 'metadata']
torch.Size([62, 400]) torch.Size([64]) torch.Size([63])


In [5]:
import h5py

HDF5_FILE = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"

with h5py.File(HDF5_FILE, "r") as f:
    print("\nDatasets in file:", list(f.keys()))

    for name in f.keys():
        dset = f[name]
        print(f"{name}: shape={dset.shape}, dtype={dset.dtype}")

    # Optional: verify a few entries
    sample_idx = 0
    print("\nSample check:")
    print("EEG sample shape:", f["eeg"][sample_idx].shape)
    print("Tokens sample shape:", f["input_ids"][sample_idx].shape)
    print("Metadata sample shape:", f["metadata"][sample_idx].shape)


Datasets in file: ['eeg', 'input_ids', 'metadata']
eeg: shape=(28000, 62, 400), dtype=float32
input_ids: shape=(28000, 64), dtype=int64
metadata: shape=(28000, 63), dtype=float32

Sample check:
EEG sample shape: (62, 400)
Tokens sample shape: (64,)
Metadata sample shape: (63,)
